## 22127260 - Bùi Công Mậu
## 22127400 - Thái Hữu Thọ

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import train_test_split
import catboost as cb
import optuna
import time
import os
import psutil

In [ ]:
# Đọc dữ liệu
train_data = pd.read_csv('../train_normalized.csv')
test_data = pd.read_csv('../test_normalized.csv')

# Xử lý ngoại lệ trong mục tiêu (loại bỏ 1% trên/dưới)
q_low = train_data["target"].quantile(0.01)
q_high = train_data["target"].quantile(0.99)
train_data = train_data[(train_data["target"] >= q_low) & (train_data["target"] <= q_high)]

# Tách đặc trưng và mục tiêu
X = train_data.drop(columns=["id", "target"])
y = train_data["target"]
X_test = test_data.drop(columns=["id"])

# Biến đổi log cho mục tiêu
y_log = np.log1p(y)

# Kỹ thuật đặc trưng
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

# Biến đổi log cho các đặc trưng số bị lệch
for col in numerical_cols:
    if X[col].skew() > 0.75:
        X[col] = np.log1p(X[col].clip(lower=0))
        X_test[col] = np.log1p(X_test[col].clip(lower=0))

# Augmentation nhẹ: Thêm 20% dữ liệu nhiễu cho đặc trưng số bị lệch
np.random.seed(42)
augment_ratio = 0.2
n_augment = int(len(X) * augment_ratio)
augment_indices = np.random.choice(X.index, size=n_augment, replace=True)
X_aug = X.loc[augment_indices].copy()
y_aug = y.loc[augment_indices].copy()

# Thêm nhiễu (4 mức: 0.005–0.02) vào đặc trưng số bị lệch
for col in numerical_cols:
    if X[col].skew() > 0.75:
        noise = np.random.uniform(0.005, 0.02) * X_aug[col].std()
        X_aug[col] += np.random.normal(0, noise, size=n_augment)

# Kết hợp dữ liệu gốc và augmentation
X = pd.concat([X, X_aug], axis=0)
y = pd.concat([y, y_aug], axis=0)
y_log = np.log1p(y)

# Theo dõi thời gian và bộ nhớ
start_time = time.time()
process = psutil.Process(os.getpid())

# Chia tập huấn luyện/validation
X_train, X_val, y_train_log, y_val_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Hàm mục tiêu cho Optuna
def objective(trial):
    # Phạm vi siêu tham số
    catboost_params = {
        'loss_function': 'RMSE',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 10, 100),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 500),
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'early_stopping_rounds': 50,
        'random_seed': 42,
        'verbose': 0
    }

    # Huấn luyện mô hình
    dtrain = cb.Pool(X_train, label=y_train_log, cat_features=categorical_cols.tolist())
    dval = cb.Pool(X_val, label=y_val_log, cat_features=categorical_cols.tolist())

    model = cb.CatBoostRegressor(**catboost_params)
    model.fit(dtrain, eval_set=dval, use_best_model=True)

    # Dự đoán và tính RMSLE
    y_pred_val_log = model.predict(X_val)
    y_pred_val = np.expm1(y_pred_val_log)
    y_val = np.expm1(y_val_log)
    val_rmsle = root_mean_squared_log_error(y_val, y_pred_val)

    return val_rmsle

# Tạo nghiên cứu Optuna và tối ưu hóa
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

# Lấy siêu tham số tốt nhất
best_params = study.best_params
print("Siêu tham số tốt nhất:", best_params)
print("RMSLE tốt nhất:", study.best_value)

# Huấn luyện lại mô hình với siêu tham số tốt nhất
best_catboost_params = {
    'loss_function': 'RMSE',
    'learning_rate': best_params['learning_rate'],
    'depth': best_params['depth'],
    'l2_leaf_reg': best_params['l2_leaf_reg'],
    'min_data_in_leaf': best_params['min_data_in_leaf'],
    'iterations': best_params['iterations'],
    'early_stopping_rounds': 50,
    'random_seed': 42,
    'verbose': 200
}

dtrain = cb.Pool(X_train, label=y_train_log, cat_features=categorical_cols.tolist())
dval = cb.Pool(X_val, label=y_val_log, cat_features=categorical_cols.tolist())

model = cb.CatBoostRegressor(**best_catboost_params)
model.fit(dtrain, eval_set=dval, use_best_model=True)

# Đánh giá trên validation
y_pred_val_log = model.predict(X_val)
y_pred_val = np.expm1(y_pred_val_log)
y_val = np.expm1(y_val_log)

# RMSLE huấn luyện
y_pred_train_log = model.predict(X_train)
y_pred_train = np.expm1(y_pred_train_log)
y_train = np.expm1(y_train_log)
train_rmsle = root_mean_squared_log_error(y_train, y_pred_train)
print(f"Train RMSLE: {train_rmsle:.4f}")

# RMSLE validation
val_rmsle = root_mean_squared_log_error(y_val, y_pred_val)
print(f"Validation RMSLE: {val_rmsle:.4f}")

# Dự đoán trên tập test
y_test_pred_log = model.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)

# Clipping chặt hơn để ổn định dự đoán
max_target = np.expm1(y_log.max())
y_test_pred = np.clip(y_test_pred, 0.01, max_target)

# Lưu submission
submission = pd.DataFrame({"id": test_data["id"], "target": y_test_pred})
submission.to_csv("22127260_22127400.csv", index=False)
print("Đã lưu file 22127260_22127400.csv với cột id và target.")

# Tính thời gian và bộ nhớ
end_time = time.time()
training_time = end_time - start_time
memory_usage = process.memory_info().rss / (1024 ** 2)

print(f"Thời gian huấn luyện: {training_time:.2f} giây")
print(f"Sử dụng bộ nhớ: {memory_usage:.2f} MB")

[I 2025-04-22 23:20:45,810] A new study created in memory with name: no-name-616f5fc0-8cd4-4b39-8a6c-c28d15e1f311
[I 2025-04-22 23:22:01,603] Trial 0 finished with value: 0.9965780119128799 and parameters: {'learning_rate': 0.027965383164977087, 'depth': 8, 'l2_leaf_reg': 99.13862807364254, 'min_data_in_leaf': 107, 'iterations': 1042}. Best is trial 0 with value: 0.9965780119128799.
[I 2025-04-22 23:23:04,473] Trial 1 finished with value: 0.9991279226682024 and parameters: {'learning_rate': 0.0195021202691549, 'depth': 5, 'l2_leaf_reg': 41.16113247587647, 'min_data_in_leaf': 155, 'iterations': 1104}. Best is trial 0 with value: 0.9965780119128799.
[I 2025-04-22 23:24:09,621] Trial 2 finished with value: 0.9986616741652014 and parameters: {'learning_rate': 0.013413827467699502, 'depth': 6, 'l2_leaf_reg': 18.81271090396809, 'min_data_in_leaf': 327, 'iterations': 1050}. Best is trial 0 with value: 0.9965780119128799.
[I 2025-04-22 23:25:22,089] Trial 3 finished with value: 0.9973318821915

Siêu tham số tốt nhất: {'learning_rate': 0.07953060127413188, 'depth': 8, 'l2_leaf_reg': 14.578753120499037, 'min_data_in_leaf': 376, 'iterations': 1341}
RMSLE tốt nhất: 0.9944029125628333
0:	learn: 1.0409698	test: 1.0409208	best: 1.0409208 (0)	total: 88.2ms	remaining: 1m 58s
200:	learn: 0.9941628	test: 0.9970903	best: 0.9970876 (199)	total: 17s	remaining: 1m 36s
400:	learn: 0.9904006	test: 0.9961404	best: 0.9961404 (400)	total: 30.9s	remaining: 1m 12s
600:	learn: 0.9868392	test: 0.9956665	best: 0.9956665 (600)	total: 44.9s	remaining: 55.3s
800:	learn: 0.9835481	test: 0.9952196	best: 0.9952164 (799)	total: 1m	remaining: 40.8s
1000:	learn: 0.9802786	test: 0.9949003	best: 0.9948957 (997)	total: 1m 16s	remaining: 25.9s
1200:	learn: 0.9770812	test: 0.9946550	best: 0.9946481 (1197)	total: 1m 31s	remaining: 10.7s
1340:	learn: 0.9749216	test: 0.9944043	best: 0.9944029 (1339)	total: 1m 43s	remaining: 0us

bestTest = 0.9944029136
bestIteration = 1339

Shrink model to first 1340 iterations.
Trai